# 02 · FunnyBirds + CBM — grounding & mechanism  *(seed-aware)*

**Claim (CBM):** a part concept reflects its part. **Backwash:** it instead reads
species / the rest of the bird. Two probes: **deletion grounding** (causal) and the
**species probe** (mechanism). All cells aggregate over every available seed
(`funnybirds-cbm-s*`); error bars are std across seeds.
*Refs: `fb_cbm_renderer_swap_v2.ipynb`, `fb_cbm_counterfactual.ipynb` §6.*

In [ ]:
import os, json, re, glob
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt
CURATED = Path(os.environ["CURATED_DATA"]); REPO = Path.cwd().parent
import sys; sys.path.insert(0, str(REPO/"analysis"))
try:
    from plotting import set_paper_style, PALETTE; set_paper_style()
    CBM_C, MCBM_C = PALETTE["CBM"], PALETTE["MCBM"]
except Exception:
    CBM_C, MCBM_C = "#0072B2", "#D55E00"
plt.rcParams["figure.dpi"]=120
EPS = 1e-3
def parse_stem(stem):
    m=re.match(r"^funnybirds-(vanilla|cbm|mcbm)(?:-g([0-9p]+))?-s(\d+)$", stem)
    if not m: return None
    gamma=float(m.group(2).replace("p",".")) if m.group(2) else np.nan
    return m.group(1), gamma, int(m.group(3))
def need(p, how):
    ok=Path(p).exists()
    if not ok: print(f"[pending] {p}\n  produce it:  {how}")
    return ok
# ---- seed-aware grounding loaders ----
def load_grounding(prefix):
    """All seeds for a config prefix -> one df with a 'seed' column (None if absent)."""
    fs = sorted(glob.glob(str(CURATED/"grounding"/f"{prefix}-s*.parquet")))
    if not fs: return None
    out=[]
    for f in fs:
        d=pd.read_parquet(f); d["seed"]=int(re.search(r"-s(\d+)\.parquet$", f).group(1)); out.append(d)
    return pd.concat(out, ignore_index=True)
def per_part_seedagg(df, visible_only=True):
    """per (seed,part) retained_frac -> per-part mean/std/count across seeds."""
    d = df[df["changed_frac"]>EPS] if (visible_only and "changed_frac" in df.columns) else df
    g = d.groupby(["seed","part"]).agg(pi=("p_intact","mean"), pr=("p_removed","mean"))
    g["rf"] = g.pr/g.pi
    return g["rf"].groupby("part").agg(["mean","std","count"])


## What a CBM is, and what `z` and `c_preds` are
`image x → encoder p(z|x) → z → concept head q(c|z) → c_preds → label head → y`
- **`z`** — per-concept bottleneck latent the encoder reads from the image (26 slots).
- **`c_preds`** — concept probabilities (concept head on `z`), the human-readable layer.
- **`y`** — the class, from `z`/concepts.
Backwash lives in **encoder→`z`**: does `z_j` read *its part's pixels* or the species?

## 0 · Training sanity & overfitting (seed 1, representative)
Per-epoch held-out accuracy from `results/funnybirds-cbm/1/predictions/epoch_*.pth`.

In [ ]:
import torch
preds = REPO/"external"/"minimal_cbm"/"results"/"funnybirds-cbm"/"1"/"predictions"
def _acc(pth):
    d = torch.load(pth, map_location="cpu", weights_only=False)
    yp,y=d["y_preds"],d["y"]
    ta=(yp.argmax(-1)==y).float().mean().item() if yp.ndim>1 else (yp==y).float().mean().item()
    ca=None
    if d.get("c_preds") is not None and d.get("c") is not None:
        cp=d["c_preds"]; cp=cp[...,0] if cp.ndim==3 else cp; ca=((cp>=0.5).float()==d["c"]).float().mean().item()
    return ta,ca
fs=sorted(glob.glob(str(preds/"epoch_*.pth")), key=lambda p:int(re.findall(r"epoch_(\d+)",p)[0]))
if not fs: print(f"[pending] no per-epoch predictions in {preds}")
else:
    R=pd.DataFrame([(int(re.findall(r"epoch_(\d+)",f)[0]),*_acc(f)) for f in fs],
                   columns=["epoch","task","concept"]).sort_values("epoch"); best=int(R.loc[R.task.idxmax(),"epoch"])
    display(R.round(4)); fig,ax=plt.subplots(figsize=(6,3.4))
    ax.plot(R.epoch,R.task,"o-",color=CBM_C,label="task (val)")
    if R.concept.notna().any(): ax.plot(R.epoch,R.concept,"s--",color="#5B8C5A",label="concept (val)")
    ax.axvline(best,ls=":",color="k"); ax.set_xlabel("epoch"); ax.set_ylabel("val acc")
    ax.set_title(f"Training curve — best task epoch {best}"); ax.legend()
    drop=R.task.max()-R.task.iloc[-1]
    print("VERDICT:", "plateau -> no overfit" if abs(drop)<0.01 else f"peaks {best} then declines")

## 1 · Deletion grounding — per-part `retained_frac` (all vs visible-only, ±seed std)
`retained_frac = P(removed)/P(intact)`. **visible-only** (headline) drops no-op removals
(part occluded in the intact image). `conf_on_occluded` = P(concept) the model asserts
for parts that are occluded in the intact image — a second, deletion-free backwash signal.

In [ ]:
G = load_grounding("funnybirds-cbm")
if G is None: print("[pending] bash analysis/grounding_sweep.sh")
else:
    allp = per_part_seedagg(G, visible_only=False)
    visp = per_part_seedagg(G, visible_only=True)
    R = pd.DataFrame({"retained_all":allp["mean"], "retained_visible":visp["mean"],
                      "vis_std":visp["std"]})
    if "changed_frac" in G.columns:
        occ = G[G["changed_frac"]<=EPS]
        R["conf_on_occluded"] = occ.groupby("part").p_intact.mean()
        R["frac_noop"] = G.groupby("part").changed_frac.apply(lambda s:(s<=EPS).mean())
    R = R.sort_values("retained_visible", ascending=False); display(R.round(3))
    print(f"n_seeds = {G.seed.nunique()}  (seeds: {sorted(G.seed.unique())})")
    fig,ax=plt.subplots(figsize=(6.6,3.4)); x=np.arange(len(R)); w=0.4
    ax.bar(x-w/2, R.retained_all, w, color="#9ec9e2", label="all removals (inflated)")
    ax.bar(x+w/2, R.retained_visible, w, yerr=R.vis_std.fillna(0), capsize=3, color=CBM_C, label="visible-only (headline)")
    ax.set_xticks(x); ax.set_xticklabels(R.index, rotation=30, ha="right")
    ax.set_ylabel("retained_frac"); ax.set_ylim(0,1); ax.legend()
    ax.set_title("FunnyBirds · CBM · removed-part concept retention")

## 2 · Species-identity probe — is the bottleneck a class code? (seed-averaged)
`species←c_preds` = how much the reported concepts alone pin the species (chance 1/50).
Per-part = species from each part's concept block alone. *(species←c_preds≈1 is partly
tautological with clean concepts; the per-part numbers are the informative ones.)*

In [ ]:
sps = sorted(glob.glob(str(CURATED/"species_probe"/"funnybirds-cbm-s*.json")))
PARTV=None
if not sps: print("[pending] bash analysis/grounding_sweep.sh")
else:
    Ss=[json.loads(Path(p).read_text()) for p in sps]; ch=Ss[0]["chance"]
    zc=np.mean([s["species_from_z"]["acc"] for s in Ss]); cc=np.mean([s["species_from_cpreds"]["acc"] for s in Ss])
    print(f"n_seeds={len(Ss)} | chance={ch:.3f} | species<-z {zc:.3f} | species<-c_preds {cc:.3f}")
    parts=list(Ss[0]["species_from_part_cpreds"].keys())
    PARTV=pd.DataFrame({"species_code":[np.mean([s["species_from_part_cpreds"][p]["acc"] for s in Ss]) for p in parts],
                        "n_variants":[Ss[0]["species_from_part_cpreds"][p]["n_variants"] for p in parts]}, index=parts)
    PARTV=PARTV.sort_values("species_code", ascending=False); display(PARTV.round(3))
    fig,ax=plt.subplots(1,2,figsize=(10,3.2))
    ax[0].bar(["z","c_preds"],[zc,cc],color=CBM_C); ax[0].axhline(ch,ls="--",color="k",label="chance")
    ax[0].set_ylim(0,1); ax[0].set_title("species recoverable from bottleneck"); ax[0].legend()
    ax[1].bar(PARTV.index, PARTV.species_code, color=CBM_C); ax[1].axhline(ch,ls="--",color="k")
    ax[1].set_title("species from EACH part's concepts"); plt.setp(ax[1].get_xticklabels(),rotation=30,ha="right")

## 3 · Line them up — does per-part retention track the mechanism?
Per part: visible-only `retained_frac` vs species-code vs n_variants (seed-averaged).
Correlation, not proof. Note wing: high species-code, low retention — species-coding is
*necessary* not *sufficient*; the part must also be the shortcut (small/hard-to-see).

In [ ]:
if G is not None and PARTV is not None:
    M = per_part_seedagg(G, visible_only=True)[["mean"]].rename(columns={"mean":"retained_frac"}).join(PARTV)
    display(M.round(3))
    fig,ax=plt.subplots(1,2,figsize=(10,3.4))
    for k,(x,xl) in enumerate({"species_code":"species from part's concepts","n_variants":"# concept variants"}.items()):
        ax[k].scatter(M[x], M.retained_frac, color=CBM_C)
        for p,r in M.iterrows(): ax[k].annotate(p,(r[x],r.retained_frac),fontsize=8)
        ax[k].set_xlabel(xl); ax[k].set_ylabel("retained_frac (visible-only)")
    plt.tight_layout(); print("tail: top on species-code AND retention = mechanism realized.")
else:
    print("[pending] needs sections 1 and 2.")

## 3c · Per-species leakage — is backwash uniform, or species-specific?
*(ports `fb_cbm_counterfactual.ipynb §5d`)* Group the **tail** deletion by species:
`retained_frac` per class. If backwash were an OOD artifact it would be uniform; if it's
species-lookup it concentrates in species whose tail is least visually distinctive.

In [ ]:
Gs = load_grounding("funnybirds-cbm")
if Gs is None or "class_idx" not in Gs.columns:
    print("[pending] grounding parquet with class_idx")
else:
    t = Gs[Gs.part=="tail"]
    if "changed_frac" in t.columns: t = t[t["changed_frac"]>EPS]
    persp = (t.groupby("class_idx").apply(lambda d: d.p_removed.mean()/d.p_intact.mean()
             if d.p_intact.mean()>1e-6 else np.nan).dropna().sort_values(ascending=False))
    persp.name="tail_retained_frac"
    print(f"tail retained_frac across {len(persp)} species: "
          f"min {persp.min():.3f}  median {persp.median():.3f}  max {persp.max():.3f}")
    print(f"top-10 species mean {persp.head(10).mean():.3f}  vs  bottom-10 {persp.tail(10).mean():.3f}")
    fig,ax=plt.subplots(1,2,figsize=(11,3.2))
    ax[0].bar(range(len(persp)), persp.values, color=CBM_C)
    ax[0].set_xlabel("species (sorted by leakage)"); ax[0].set_ylabel("tail retained_frac")
    ax[0].set_title("Per-species tail retained_frac — NOT uniform")
    ax[1].hist(persp.values, bins=20, color=CBM_C); ax[1].set_xlabel("tail retained_frac"); ax[1].set_ylabel("# species")
    ax[1].set_title("Distribution across species"); plt.tight_layout()
    print("Non-uniform across species -> species-specific leakage, not global OOD confusion.")

## 5 · Pre-swap donor activation — backwash with NO swap  *(ref §42, elevated)*
The cleanest signature, needing no counterfactual: on the **original** image, how much
does the *donor* concept (a part variant the bird does **not** have) already fire?
`z_donor_absent` should be ≪ `z_src_present` if the bottleneck reads pixels; if it's
high, the concept is pre-activated from species identity = backwash. (Reads the swap CSV
columns `z_old_orig` = source concept present, `z_new_orig` = donor concept absent.)

In [ ]:
SW = CURATED/"swap"/"funnybirds-cbm-s1.csv"
ORDER=["tail","wing","beak","foot","eye"]
if not need(SW, 'CONFIG_PREFIX=funnybirds-cbm GAMMAS="0" SEEDS="1" sbatch train/renderer_swap.slurm'):
    S=None
else:
    S=pd.read_csv(SW)
    g=S.groupby("part").agg(z_src_present=("z_old_orig","mean"),
                            z_donor_absent=("z_new_orig","mean")).reindex(ORDER)
    display(g.round(3))
    fig,ax=plt.subplots(figsize=(6.6,3.3)); x=np.arange(len(g)); w=0.4
    ax.bar(x-w/2,g.z_src_present,w,color="#5B8C5A",label="source concept (part PRESENT)")
    ax.bar(x+w/2,g.z_donor_absent,w,color=CBM_C,label="donor concept (part ABSENT)")
    ax.axhline(0,color="k",lw=0.5); ax.set_xticks(x); ax.set_xticklabels(g.index)
    ax.set_ylabel("mean concept logit (original image)"); ax.legend(fontsize=8)
    ax.set_title("Pre-swap donor activation — high 'absent' bar = backwash")

## 6 · Renderer-swap z-ordering — the causal test  *(ref §7)*
Re-render each image with one part swapped to another species' variant (same camera /
light / background). `ordering_correct = z_cf[donor] > z_cf[src]` — grounded ⇒ the
swapped-in part wins ⇒ ~1. We keep **fwd and bwd separate** (averaging them hides the
signal). Violations = the bottleneck stayed anchored to the source species = backwash.

In [ ]:
if S is not None:
    ordf=S.groupby(["part","direction"]).ordering_correct.mean().unstack().reindex(ORDER)
    display(ordf.round(3))
    fig,ax=plt.subplots(figsize=(6.6,3.3)); x=np.arange(len(ordf)); w=0.4
    ax.bar(x-w/2,ordf.get("fwd"),w,color=CBM_C,label="fwd (A's image ← B's part)")
    ax.bar(x+w/2,ordf.get("bwd"),w,color="#D55E00",label="bwd (B's image ← A's part)")
    ax.axhline(1.0,ls=":",color="green",label="grounded (=1)"); ax.axhline(0.5,ls=":",color="gray")
    ax.set_xticks(x); ax.set_xticklabels(ordf.index); ax.set_ylim(0,1.05)
    ax.set_ylabel("ordering_correct (donor concept wins after swap)"); ax.legend(fontsize=8)
    ax.set_title("Renderer-swap z-ordering per part (fwd/bwd)")
    print("~1 = swapped-in part detected (grounded). <1 = anchored to source species = backwash.")
else: print("[pending] swap CSV")

### 6b · Per-swap margin dots (fwd=blue, bwd=red)  *(ref §43–45)*
Every individual swap's `margin = z_cf[donor] − z_cf[src]`. Dots **below 0** are
ordering violations (the source concept still won after the part was swapped out) =
backwash. Kept per-direction so you see the raw spread, not just the mean.

In [ ]:
if S is not None:
    rs=np.random.RandomState(0)
    fig,ax=plt.subplots(figsize=(8,4))
    for j,part in enumerate(ORDER):
        d=S[S.part==part]
        for dirn,col in [("fwd","#1f77b4"),("bwd","#d62728")]:
            dd=d[d.direction==dirn]
            xj=rs.normal(j+(-0.15 if dirn=="fwd" else 0.15),0.05,len(dd))
            ax.scatter(xj,dd.margin,s=7,alpha=0.35,color=col,label=dirn if j==0 else None)
    ax.axhline(0,color="k",lw=1); ax.set_xticks(range(len(ORDER))); ax.set_xticklabels(ORDER)
    ax.set_ylabel("z-ordering margin (donor − source)"); ax.legend(title="direction")
    ax.set_title("Per-swap margin — dots below 0 = violations = backwash")
else: print("[pending] swap CSV")

### 6c · Inspection grid — only the target part changes  *(ref §28/§62)*
Qualitative check: for each part, the original render, the swap (one part → donor
variant), and the deletion. Camera / light / background are identical across columns —
only the part changes. (Displays the example PNGs the swap SLURM job saved to
`swap/examples/`; no live renderer needed here.)

In [ ]:
import matplotlib.image as mpimg
exdir=CURATED/"swap"/"examples"; tags=["orig","swap","delete"]
if exdir.exists() and any(exdir.glob("*.png")):
    fig,axes=plt.subplots(len(ORDER),len(tags),figsize=(3*len(tags),2.7*len(ORDER)))
    for r,part in enumerate(ORDER):
        for c,t in enumerate(tags):
            axm=axes[r,c] if len(ORDER)>1 else axes[c]
            fs=sorted(exdir.glob(f"{part}_*_{t}.png"))
            if fs: axm.imshow(mpimg.imread(str(fs[0])))
            axm.set_title(f"{part} · {t}",fontsize=8); axm.axis("off")
    plt.tight_layout()
else:
    need(exdir/"<part>_*_orig.png", 'run train/renderer_swap.slurm (it saves examples first)')

## 7 · Occlusion control — is it visibility, or backwash?  *(ref §72, the make-or-break)*
A z-ordering violation could be trivial: the renderer sometimes places the swapped-in
part behind other geometry (few visible pixels), so the model *can't* see it. We filter
to swaps where the part is actually visible (`pixel_count_cf ≥ 50`) and re-check. **If
`ordering_correct` rises after filtering → occlusion explained it. If it stays low →
backwash.**

In [ ]:
if S is not None and "pixel_count_cf" in S.columns:
    MIN_PX=50
    before=S.groupby("part").ordering_correct.mean()
    after =S[S.pixel_count_cf>=MIN_PX].groupby("part").ordering_correct.mean()
    C=pd.DataFrame({"all_swaps":before,f"visible_only(>={MIN_PX}px)":after}).reindex(ORDER)
    display(C.round(3))
    fig,ax=plt.subplots(1,2,figsize=(11,3.4)); x=np.arange(len(C)); w=0.4
    ax[0].bar(x-w/2,C.iloc[:,0],w,color="#9ec9e2",label="all swaps")
    ax[0].bar(x+w/2,C.iloc[:,1],w,color=CBM_C,label="visible-only")
    ax[0].axhline(1,ls=":",color="green"); ax[0].set_xticks(x); ax[0].set_xticklabels(C.index)
    ax[0].set_ylim(0,1.05); ax[0].set_ylabel("ordering_correct"); ax[0].legend(fontsize=8)
    ax[0].set_title("Filter low-visibility swaps")
    t=S[S.part=="tail"]
    ax[1].scatter(t.pixel_count_cf,t.margin,s=6,alpha=0.3,color=CBM_C)
    ax[1].axhline(0,color="k",lw=0.5); ax[1].axvline(MIN_PX,ls=":",color="crimson")
    ax[1].set_xlabel("swapped-in tail pixels (visibility)"); ax[1].set_ylabel("margin")
    ax[1].set_title("tail: does low visibility predict violations?")
    print("Rises after filtering -> occlusion. Stays low -> backwash (the real result).")
elif S is not None:
    print("[pending] swap CSV lacks pixel_count_cf (run WITHOUT --no-v2)")

## 8 · Concept confusion after swap (tail)  *(ref §50)*
When a tail swap is mis-ordered, which tail concept fires strongest on the swapped
image? Row = the donor variant that *should* win; column = the variant that actually
fires. Off-diagonal mass = the swapped-in tail is mis-attributed to the wrong slot.

In [ ]:
if S is not None:
    tcols=[c for c in S.columns if c.startswith("z_cf_tail_")]
    t=S[S.part=="tail"].copy()
    if tcols and len(t):
        Z=t[tcols].values; predv=Z.argmax(1); n=len(tcols)
        Mc=np.zeros((n,n))
        for dv,pv in zip(t.var_donor.astype(int),predv): Mc[dv,pv]+=1
        Mc=Mc/Mc.sum(1,keepdims=True).clip(min=1)
        fig,ax=plt.subplots(figsize=(5,4.3)); im=ax.imshow(Mc,cmap="magma",vmin=0,vmax=1)
        ax.set_xlabel("tail concept that fires strongest (argmax)"); ax.set_ylabel("donor tail variant (should win)")
        ax.set_title("Tail concept confusion after swap (diagonal=grounded)"); fig.colorbar(im,fraction=0.046)
        print("diagonal =", round(float(np.mean(predv==t.var_donor.astype(int).values)),3), "of tail swaps correctly attributed")
    else: print("[pending] no tail z_cf columns in swap CSV")
else: print("[pending] swap CSV")

## Takeaway
CBM concepts near-perfect on-distribution, yet a **removed** part's concept is retained
(tail), concentrated where the bottleneck most encodes species AND the part is hardest to
see. Notebook 03 asks whether MCBM minimality removes this.

## How `retained_frac` is read as a backwash measurement
No axis is labelled "backwash"; the computed number is
**`retained_frac = P(concept | part removed) / P(concept | intact)`**, on **visible-only**
removals (the part actually left the render). Grounded → collapses to ~0; backwashed →
stays ~1. `retained_frac` is the metric; "concept–class backwash" is the interpretation.